# Cyclone Intensity Estimation: Fine-Tuning ResNet50
This notebook demonstrates how to download a subset of the **Digital Typhoon Dataset** (or an IMD image dataset) and fine-tune a pre-trained PyTorch ResNet50 model for cyclone intensity classification.

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import os

print('PyTorch Version:', torch.__version__)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

### 1. Data Preparation
Normally, you would download the Digital Typhoon metadata and images. For this example, we assume you have structured the dataset into `data/train/` and `data/val/` with subfolders for each category (e.g. `CS`, `SCS`, `VSCS`, `ESCS`, `SuCS`).

In [ ]:
# Define transforms (ResNet50 expects 224x224)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# image_datasets = {x: datasets.ImageFolder(os.path.join('data', x), data_transforms[x]) for x in ['train', 'val']}
# dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=4) for x in ['train', 'val']}

### 2. Model Setup
We load a pre-trained ResNet50 model and replace the final fully-connected layer to match our number of cyclone categories.

In [ ]:
model = models.resnet50(pretrained=True)

# Freeze early layers (optional, but good for small datasets)
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer
num_ftrs = model.fc.in_features
num_classes = 5 # CS, SCS, VSCS, ESCS, SuCS
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

### 3. Training Loop
A standard PyTorch training loop for fine-tuning.

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    print("Starting training loop...")
    # for epoch in range(num_epochs):
        # Iterate over dataloaders['train'] and dataloaders['val']
        # compute loss, backpropagate, step optimizer
        # ...
    print("Training complete.")
    return model

# model = train_model(model, criterion, optimizer, num_epochs=15)
# torch.save(model.state_dict(), 'cyclone_resnet50_finetuned.pt')